# **BASELINE MODEL SELECTION**

We selected Logistic Regression and Random Forest as the two baseline models to evaluate and compare their performance before developing the final model presented in `5.Model.ipynb`.

Logistic Regression was chosen because it is a simple and interpretable classification model that serves as a strong baseline for comparison. Meanwhile, Random Forest was selected because it is capable of capturing more complex and non-linear relationships within the dataset. By comparing these two models, we were able to evaluate how different modeling approaches performed on the imbalanced classification problem.


### **RELOAD DATA & LIBRARY**

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    f1_score,
    roc_auc_score
)
# for save and load model
import joblib
import json
import os

#Show all columns, without hiding any
pd.set_option('display.max_columns', None)

In [8]:
#Load the dataset
path = "../Data/Processed/data_cleaned.csv"
df = pd.read_csv(path)

#Quick look at the data
df.head()

,track_number,track_popularity,explicit,artist_popularity,artist_followers,artist_genres,album_id,album_total_tracks,album_type,track_duration_min,release_year,release_month
0,4,0,True,77,2812821,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.55,2025,10
1,1,0,True,64,2363438,"country hip hop, southern hip hop",4SUmmwnv0xTjRcLdjczGg2,1,single,3.07,2025,10
2,1,4,True,48,193302,Unknown,3E3zEAL8gUYWaLYB9L7gbp,1,single,2.55,2025,10
3,8,30,True,77,2813710,moombahton,5QRFnGnBeMGePBKF2xTz5z,9,album,1.69,2025,10
4,2,0,True,48,8682,dark r&b,06FDIpSHYmZAZoyuYtc7kd,2,single,2.39,2025,10


### ***1. LOGISTIC REGRESSION***

In [9]:


# =========================
# READ DATA
# =========================

df = pd.read_csv(r'..\\Data\\Processed\\data_cleaned.csv')

# check shape
print(df.shape)

# =========================
# CREATE BINARY TARGET
# =========================

def hit_or_not(x):

    if x > 80:
        return 1   # Hit
    else:
        return 0   # Not Hit

df['target'] = df['track_popularity'].apply(hit_or_not)

# =========================
# SELECT NUMERIC COLUMNS
# =========================

numeric_df = df.select_dtypes(
    include=['int64', 'float64']
)

# =========================
# CREATE X and y
# =========================

X = numeric_df.drop(
    columns=['track_popularity']
)

y = df['target']

# =========================
# CHECK
# =========================

print(X.shape)
print(y.shape)

print("\nTarget Distribution:\n")

print(
    y.value_counts()
)

# =========================
# TRAIN TEST SPLIT
# =========================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# LOGISTIC REGRESSION MODEL
# =========================

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000
)

# =========================
# TRAIN MODEL
# =========================

model.fit(X_train, y_train)

# =========================
# PREDICT
# =========================

y_pred = model.predict(X_test)

# =========================
# EVALUATE
# =========================

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print("\nAccuracy:")

print(
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:\n")

print(
    classification_report(y_test, y_pred)
)

(8582, 12)
(8582, 8)
(8582,)

Target Distribution:

target
0    7990
1     592
Name: count, dtype: int64

Accuracy:
0.9312754804892254

Classification Report:

              precision    recall  f1-score   support

           0       0.93      1.00      0.96      1599
           1       0.00      0.00      0.00       118

    accuracy                           0.93      1717
   macro avg       0.47      0.50      0.48      1717
weighted avg       0.87      0.93      0.90      1717



### ***2. RANDOM FOREST***

In [10]:

# =====================================
# 1. CREATE BINARY TARGET
# =====================================

def hit_or_not(x):

    if x > 80:
        return "Hit"
    else:
        return "Not Hit"

df['target'] = df['track_popularity'].apply(hit_or_not)

# =====================================
# 2. CREATE X (FEATURES)
# =====================================

# select numeric columns
X = df.select_dtypes(include=['int64', 'float64'])

# drop the popularity column to avoid data leakage
X = X.drop(columns=['track_popularity'])

# =====================================
# 3. CREATE y (TARGET)
# =====================================

y = df['target']

# =====================================
# 4. TRAIN TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================
# 5. RANDOM FOREST BASELINE MODEL
# =====================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# =====================================
# 6. TRAIN MODEL
# =====================================

rf_model.fit(X_train, y_train)

# =====================================
# 7. PREDICT
# =====================================

y_pred = rf_model.predict(X_test)

# =====================================
# 8. EVALUATE MODEL
# =====================================

print("Random Forest Accuracy:")

print(
    accuracy_score(y_test, y_pred)
)

print("\nClassification Report:\n")

print(
    classification_report(y_test, y_pred)
)

# =====================================
# 9. CHECK TARGET DISTRIBUTION
# =====================================

print("\nTarget Distribution:\n")

print(
    df['target'].value_counts()
)

Random Forest Accuracy:
0.930110658124636

Classification Report:

              precision    recall  f1-score   support

         Hit       0.46      0.09      0.15       118
     Not Hit       0.94      0.99      0.96      1599

    accuracy                           0.93      1717
   macro avg       0.70      0.54      0.56      1717
weighted avg       0.90      0.93      0.91      1717


Target Distribution:

target
Not Hit    7990
Hit         592
Name: count, dtype: int64


### **CONCLUSION**

The first baseline model achieved a high accuracy of 93%, but failed to detect any “Hit” cases, resulting in a recall and F1-score of 0.00 for the minority class. This indicates that the model mainly predicted the majority class (“Not Hit”) due to the imbalanced dataset.

In comparison, the Random Forest model achieved a similar accuracy while being able to identify some “Hit” cases, with higher precision, recall, and F1-score for the minority class. Although the performance is still limited, Random Forest showed better capability in learning patterns from the target class.

Additionally, since approximately 93% of the dataset belongs to the “Not Hit” class, both baseline models naturally achieved high accuracy by mainly predicting the majority class. Therefore, accuracy alone was not a reliable metric for evaluating model performance in this imbalanced classification problem.

Therefore, Random Forest was selected as the final model because it provided more balanced and meaningful classification performance.